# Feature Engineering

This notebook creates customer-level feature tables for the customer segmentation project.

It starts from `customer_info`, then adds simple customer-level features derived from `customer_basket`. The final clustering baseline feature selection is handled later in the advanced EDA notebook.

## Imports

Use direct data loading in the notebook and import reusable feature logic from `src/features.py`.

In [1]:
import sys

import pandas as pd

sys.path.append("../src")

from features import (
    add_customer_info_features,
    create_basket_features,
    get_lifetime_spend_columns,
)

## Load Customer Info

Load the raw customer-level data directly from `data/raw`.

In [2]:
customer_info = pd.read_csv("../data/raw/customer_info.csv")
customer_info.head()

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
0,3,Bsc. Crystal Kitchens,female,02/12/1970 01:36 PM,1.0,1.0,1.0,3.0,11731.0,4553.0,...,213.0,552.0,256.0,384.0,189.0,0.631599,2020.0,1.0,38.794428,-9.215739
1,4,Bsc. Glenda Bauman,female,11/13/1975 06:58 PM,1.0,0.0,0.0,2.0,13694.0,963.0,...,15.0,1880.0,333.0,665.0,130.0,0.149890,2013.0,1.0,38.751711,-9.179611
2,5,Msc. Antonio Campbell,male,09/10/1971 10:07 AM,0.0,0.0,NaN,2.0,12407.0,0.0,...,273.0,507.0,101.0,222.0,81.0,0.069126,2005.0,NaN,38.780678,-9.160656
3,7,John Kelling,male,10/23/1982 11:20 AM,0.0,0.0,2.0,1.0,7493.0,1105.0,...,1083.0,485.0,1656.0,184.0,92.0,0.253609,2021.0,1.0,38.739548,-9.148679
4,8,Arthur Dematteo,male,08/04/1969 10:22 PM,0.0,0.0,3.0,1.0,9187.0,10841.0,...,1015.0,297.0,1258.0,441.0,6.0,0.186569,2021.0,1.0,38.733071,-9.188188


## Reference Year Check

Use the dataset reference year from `year_first_transaction` for both `age` and `customer_tenure`. This avoids using the current calendar year and keeps tenure aligned with the dataset.

In [3]:
reference_year = customer_info["year_first_transaction"].max()

year_reference_check = pd.DataFrame(
    {
        "metric": [
            "dataset reference year used",
            "minimum year_first_transaction",
            "maximum year_first_transaction",
        ],
        "value": [
            reference_year,
            customer_info["year_first_transaction"].min(),
            customer_info["year_first_transaction"].max(),
        ],
    }
)

year_reference_check

,metric,value
0,dataset reference year used,2029.0
1,minimum year_first_transaction,1993.0
2,maximum year_first_transaction,2029.0


The feature function uses the maximum `year_first_transaction` in the dataset as the reference year. This is a dataset-based choice, not a current-calendar-year choice.

## Create Customer Info Features

Apply the reusable feature function to build the customer-level feature table.

In [4]:
customer_features_info = add_customer_info_features(customer_info)
customer_features_info.head()

,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,degree_level
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.244917,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.013771,0.020656,Bsc
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.047596,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.016458,0.032867,Bsc
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.000000,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.006496,0.014277,Msc
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.073903,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.110754,0.012306,Unknown
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.420243,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.048765,0.017095,Unknown


In [5]:
tenure_validation = pd.DataFrame(
    {
        "metric": [
            "reference_year_used",
            "customer_tenure_min",
            "customer_tenure_max",
            "customer_tenure_mean",
            "customers_with_negative_tenure",
        ],
        "value": [
            reference_year,
            customer_features_info["customer_tenure"].min(),
            customer_features_info["customer_tenure"].max(),
            customer_features_info["customer_tenure"].mean(),
            (customer_features_info["customer_tenure"] < 0).sum(),
        ],
    }
)

tenure_validation

,metric,value
0,reference_year_used,2029.000000
1,customer_tenure_min,0.000000
2,customer_tenure_max,36.000000
3,customer_tenure_mean,13.688147
4,customers_with_negative_tenure,0.000000


## Feature Columns

Review the generated spend share columns and the final feature table columns.

In [6]:
lifetime_spend_columns = get_lifetime_spend_columns(customer_info)
spend_share_columns = [
    f"share_{column.replace('lifetime_spend_', '')}"
    for column in lifetime_spend_columns
]

print("Lifetime spend columns used:")
print(lifetime_spend_columns)
print("\nSpend share columns created:")
print(spend_share_columns)

Lifetime spend columns used:
['lifetime_spend_groceries', 'lifetime_spend_electronics', 'lifetime_spend_vegetables', 'lifetime_spend_nonalcohol_drinks', 'lifetime_spend_alcohol_drinks', 'lifetime_spend_meat', 'lifetime_spend_fish', 'lifetime_spend_hygiene', 'lifetime_spend_videogames', 'lifetime_spend_petfood']

Spend share columns created:
['share_groceries', 'share_electronics', 'share_vegetables', 'share_nonalcohol_drinks', 'share_alcohol_drinks', 'share_meat', 'share_fish', 'share_hygiene', 'share_videogames', 'share_petfood']


In [7]:
customer_features_info.columns.to_frame(index=False, name="column")

,column
0,customer_id
1,customer_gender
2,kids_home
3,teens_home
4,number_complaints
5,distinct_stores_visited
6,lifetime_total_distinct_products
7,year_first_transaction
8,percentage_of_products_bought_promotion
9,typical_hour


## Basic Validation

Check that the feature table remains one row per customer and keeps all customers from `customer_info`.

In [8]:
feature_validation_checks = pd.DataFrame(
    {
        "check": [
            "customer_features_info rows",
            "customer_features_info columns",
            "unique customer_id count",
            "duplicated customer_id count",
            "one row per customer_id",
        ],
        "value": [
            customer_features_info.shape[0],
            customer_features_info.shape[1],
            customer_features_info["customer_id"].nunique(dropna=False),
            customer_features_info["customer_id"].duplicated().sum(),
            customer_features_info["customer_id"].is_unique,
        ],
    }
)

feature_validation_checks

,check,value
0,customer_features_info rows,33038
1,customer_features_info columns,29
2,unique customer_id count,33038
3,duplicated customer_id count,0
4,one row per customer_id,True


In [9]:
customer_features_info.head()

,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,degree_level
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.244917,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.013771,0.020656,Bsc
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.047596,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.016458,0.032867,Bsc
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.000000,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.006496,0.014277,Msc
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.073903,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.110754,0.012306,Unknown
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.420243,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.048765,0.017095,Unknown


## Missing Values in Engineered Features

Inspect missing values in the newly engineered columns. Missing values are not fully resolved yet because preprocessing will happen later.

In [10]:
engineered_feature_columns = [
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "has_children",
    "total_lifetime_spend",
    *spend_share_columns,
    "degree_level",
]

engineered_missing_summary = pd.DataFrame(
    {
        "missing_count": customer_features_info[engineered_feature_columns].isna().sum(),
        "missing_percentage": customer_features_info[engineered_feature_columns].isna().mean() * 100,
    }
).sort_values("missing_percentage", ascending=False)

engineered_missing_summary

,missing_count,missing_percentage
share_fish,991,2.999576
share_vegetables,661,2.000726
share_petfood,661,2.000726
share_videogames,661,2.000726
share_electronics,661,2.000726
share_meat,661,2.000726
total_children_home,656,1.985592
share_alcohol_drinks,330,0.998850
share_hygiene,330,0.998850
age,165,0.499425


## Summary Statistics

Review the new numerical features before saving the intermediate table.

In [11]:
new_numerical_features = [
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "has_children",
    "total_lifetime_spend",
    *spend_share_columns,
]

customer_features_info[new_numerical_features].describe()

,age,customer_tenure,has_loyalty_card,total_children_home,has_children,total_lifetime_spend,share_groceries,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood
count,32873.000000,33038.000000,33038.000000,32382.000000,33038.000000,33038.000000,33038.000000,32377.000000,32377.000000,33038.000000,32708.000000,32377.000000,32047.000000,32708.000000,32377.000000,32377.000000
mean,58.120251,13.688147,0.603305,2.015255,0.841728,23614.808251,0.650852,0.116794,0.042677,0.022754,0.029597,0.035638,0.028005,0.043126,0.018364,0.018400
std,18.064552,5.032196,0.489219,1.783107,0.365002,13363.992878,0.186576,0.128472,0.052503,0.015621,0.025329,0.030074,0.024078,0.040840,0.023978,0.014309
min,27.000000,0.000000,0.000000,0.000000,0.000000,2083.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,42.000000,10.000000,0.000000,1.000000,1.000000,14391.250000,0.573827,0.032752,0.009490,0.012194,0.011085,0.015552,0.009859,0.016895,0.005957,0.009101
50%,58.000000,14.000000,1.000000,2.000000,1.000000,20179.500000,0.703994,0.075370,0.021640,0.018983,0.022656,0.031136,0.023531,0.031112,0.010144,0.015112
75%,74.000000,17.000000,1.000000,2.000000,1.000000,29093.750000,0.782268,0.144775,0.057152,0.028887,0.041730,0.047368,0.038744,0.053835,0.018267,0.023601
max,89.000000,36.000000,1.000000,14.000000,1.000000,116366.000000,0.950785,0.827987,0.552543,0.200375,0.446876,0.493976,0.311507,0.445301,0.290850,0.236300


In [12]:
customer_features_info["degree_level"].value_counts(dropna=False)

degree_level
Unknown    17731
Bsc         5154
Phd         5096
Msc         5057
Name: count, dtype: int64

In [13]:
customer_features_info["has_loyalty_card"].value_counts(dropna=False)

has_loyalty_card
1    19932
0    13106
Name: count, dtype: int64

## Save Intermediate Customer Feature Table

Save the unscaled, interpretable customer-level feature table based only on `customer_info`. This is not a model-ready scaled dataset.

In [14]:
customer_features_info.to_csv("../data/processed/customer_features_info.csv", index=False)
print("Saved ../data/processed/customer_features_info.csv")

Saved ../data/processed/customer_features_info.csv


## Notes

- `customer_name` is not kept raw because names are identifiers, not useful modelling inputs; only the simple `degree_level` prefix signal is retained.
- `loyalty_card_number` is not kept raw because it behaves like an indicator; it is represented as `has_loyalty_card`.
- Spend shares were created to describe each customer's spending mix relative to total lifetime spend.
- This table is not model-ready yet because missing values, invalid values, encoding, scaling, and feature selection still need to be handled.
- Basket-derived features are added in the next section.
- Preprocessing will happen later, after the full customer-level feature table is assembled.

# Customer Basket Feature Engineering

Add simple customer-level features derived from `customer_basket`. These features are based only on the sampled basket records and are merged into the existing customer-info feature table.

## Load Customer Basket and Customer Info Features

Load `customer_basket` directly from the raw data folder and reload the existing customer-info feature table from `data/processed`.

In [15]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_basket.head()

,invoice_id,list_of_goods,customer_id
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807


In [16]:
customer_features_info = pd.read_csv("../data/processed/customer_features_info.csv")
customer_features_info.head()

,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,degree_level
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.244917,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.013771,0.020656,Bsc
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.047596,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.016458,0.032867,Bsc
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.000000,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.006496,0.014277,Msc
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.073903,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.110754,0.012306,Unknown
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.420243,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.048765,0.017095,Unknown


## Create Basket-Derived Features

Use reusable functions from `src/features.py` to parse `list_of_goods`, compute basket sizes, and aggregate basket behavior by `customer_id`.

In [17]:
basket_features = create_basket_features(customer_basket)
basket_features.head()

,customer_id,basket_count,avg_basket_size,max_basket_size,min_basket_size,total_items_bought_in_baskets,distinct_products_in_baskets,avg_distinct_products_per_basket,most_frequent_product
0,3,2,10.5,11,10,21,21,10.5,asparagus
1,4,2,12.0,12,12,24,20,12.0,airpods
2,5,1,8.0,8,8,8,8,8.0,bramble
3,7,1,4.0,4,4,4,4,4.0,cream
4,9,2,10.0,14,6,20,20,10.0,babies food


In [18]:
basket_features.shape

(28127, 9)

## Merge Basket Features

Left merge basket features into the customer-info feature table so every customer remains in the final dataset.

In [19]:
customer_features = customer_features_info.merge(basket_features, on="customer_id", how="left")

basket_numeric_features = [
    "basket_count",
    "avg_basket_size",
    "max_basket_size",
    "min_basket_size",
    "total_items_bought_in_baskets",
    "distinct_products_in_baskets",
    "avg_distinct_products_per_basket",
]

customer_features[basket_numeric_features] = customer_features[basket_numeric_features].fillna(0)

basket_count_columns = [
    "basket_count",
    "max_basket_size",
    "min_basket_size",
    "total_items_bought_in_baskets",
    "distinct_products_in_baskets",
]
customer_features[basket_count_columns] = customer_features[basket_count_columns].astype(int)

customer_features["most_frequent_product"] = customer_features["most_frequent_product"].fillna("No Basket")
customer_features.head()

,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_petfood,degree_level,basket_count,avg_basket_size,max_basket_size,min_basket_size,total_items_bought_in_baskets,distinct_products_in_baskets,avg_distinct_products_per_basket,most_frequent_product
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.020656,Bsc,2,10.5,11,10,21,21,10.5,asparagus
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.032867,Bsc,2,12.0,12,12,24,20,12.0,airpods
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.014277,Msc,1,8.0,8,8,8,8,8.0,bramble
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.012306,Unknown,1,4.0,4,4,4,4,4.0,cream
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.017095,Unknown,0,0.0,0,0,0,0,0.0,No Basket


## Basket Feature Validation

Check that the merged customer-level table preserves all customers and has no duplicate customer IDs.

In [20]:
basket_feature_validation_checks = pd.DataFrame(
    {
        "check": [
            "basket_features rows",
            "basket_features columns",
            "customer_features rows",
            "customer_features columns",
            "customer_features row count equals 33,038",
            "unique customer_id count",
            "one row per customer_id",
            "duplicated customer_id count",
            "customers with basket_count == 0",
        ],
        "value": [
            basket_features.shape[0],
            basket_features.shape[1],
            customer_features.shape[0],
            customer_features.shape[1],
            customer_features.shape[0] == 33038,
            customer_features["customer_id"].nunique(dropna=False),
            customer_features["customer_id"].is_unique,
            customer_features["customer_id"].duplicated().sum(),
            (customer_features["basket_count"] == 0).sum(),
        ],
    }
)

basket_feature_validation_checks

,check,value
0,basket_features rows,28127
1,basket_features columns,9
2,customer_features rows,33038
3,customer_features columns,37
4,"customer_features row count equals 33,038",True
5,unique customer_id count,33038
6,one row per customer_id,True
7,duplicated customer_id count,0
8,customers with basket_count == 0,4911


In [21]:
customer_features[basket_numeric_features].describe()

,basket_count,avg_basket_size,max_basket_size,min_basket_size,total_items_bought_in_baskets,distinct_products_in_baskets,avg_distinct_products_per_basket
count,33038.000000,33038.000000,33038.000000,33038.000000,33038.000000,33038.000000,33038.000000
mean,3.026818,7.973883,10.316726,5.631757,28.451329,23.759035,7.973883
std,2.854824,4.121901,5.373790,3.922077,27.939597,20.257936,4.121901
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,6.229167,7.000000,3.000000,9.000000,9.000000,6.229167
50%,2.000000,9.000000,12.000000,5.000000,21.000000,20.000000,9.000000
75%,4.000000,10.666667,14.000000,8.000000,41.000000,35.000000,10.666667
max,33.000000,18.000000,18.000000,18.000000,324.000000,145.000000,18.000000


In [22]:
customer_features["most_frequent_product"].value_counts(dropna=False).head(20)

most_frequent_product
No Basket               4911
airpods                 3317
asparagus               2510
babies food             1116
almonds                  966
antioxydant juice        916
butter                   899
bacon                    771
cereals                  690
avocado                  614
bluetooth headphones     539
black tea                488
beer                     485
body spray               480
barbecue sauce           442
cooking oil              438
eggs                     405
deodorant                400
dog food                 379
brownies                 369
Name: count, dtype: int64

In [23]:
customer_features.isna().sum().sort_values(ascending=False)

share_fish                                 991
number_complaints                          661
share_meat                                 661
typical_hour                               661
share_videogames                           661
share_vegetables                           661
share_electronics                          661
share_petfood                              661
total_children_home                        656
kids_home                                  330
teens_home                                 330
distinct_stores_visited                    330
share_alcohol_drinks                       330
percentage_of_products_bought_promotion    330
share_hygiene                              330
age                                        165
basket_count                                 0
degree_level                                 0
avg_basket_size                              0
max_basket_size                              0
min_basket_size                              0
total_items_b

## Save Final Customer Feature Table

Save the unscaled customer-level feature table with customer_info and customer_basket-derived features.

In [24]:
customer_features.to_csv("../data/processed/customer_features.csv", index=False)
print("Saved ../data/processed/customer_features.csv")

Saved ../data/processed/customer_features.csv


## Basket Feature Notes

- Basket features are based on the sampled baskets only.
- Customers without basket records are kept and receive `0` for numeric basket features.
- `most_frequent_product` is useful for profiling, but may not be used directly in clustering.
- The dataset is still not preprocessed, scaled, or model-ready.
- Preprocessing and feature selection will happen later.